In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
# Load Traning & Testing Dataset

train_df = pd.read_csv(
    "../data/processed/train_feature_engineered.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_feature_engineered.csv"
)

print("Training Shape:", train_df.shape)
print("Testing Shape:", test_df.shape)

train_df.head()

Training Shape: (1460, 87)
Testing Shape: (1459, 86)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,YrSold,SaleType,SaleCondition,SalePrice,HouseAge,RemodAge,TotalArea,TotalBathrooms,TotalPorchArea,QualityCategory
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,2008,WD,Normal,208500,5,5,2566,3.5,61,Medium
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,2007,WD,Normal,181500,31,31,2524,2.5,298,Medium
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,2008,WD,Normal,223500,7,6,2706,3.5,42,Medium
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,2006,WD,Abnorml,140000,91,36,2473,2.0,307,Medium
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,2008,WD,Normal,250000,8,8,3343,3.5,276,High


In [3]:
selected_model = joblib.load(
    "../models/tuned_random_forest_model.pkl"
)

print("Selected Model Loaded Successfully!")

Selected Model Loaded Successfully!


In [5]:
# To Seprate The Feature & Target

X = train_df.drop(
    columns=["SalePrice", "Id"]
)

y = train_df["SalePrice"]

X_test = test_df.drop(
    columns=["Id"]
)

test_ids = test_df["Id"]

print("Training Features:", X.shape)
print("Training Target:", y.shape)
print("Testing Features:", X_test.shape)

Training Features: (1460, 85)
Training Target: (1460,)
Testing Features: (1459, 85)


# Logic:

X: House information used for the model to learn.

y: Actual house prices that the model should learn.

X_test: Houses whose prices need to be predicted.

test_ids: Original ID of each house, which is required in the submission.

In [7]:
# The Final Model Train on Traning Data

selected_model.fit(X, y)

print("Final Model Training Completed!")

Final Model Training Completed!


# fter checking the model's performance during validation, it is beneficial to use the entire available labeled dataset for making final predictions.

Since it is a Pipeline, imputation, scaling, and encoding will be fitted again on the entire training data.

In [9]:
# Predicition on Kaggle Test Dataset

final_predictions = selected_model.predict(X_test)

print("First 10 Predictions:")
print(final_predictions[:10])

print("Total Predictions:", len(final_predictions))

First 10 Predictions:
[125763.508 162952.164 183611.082 186284.34  196178.534 182321.464
 176294.2   175421.794 187739.434 130890.28 ]
Total Predictions: 1459


In [10]:
# To Check Prediction 

prediction_summary = pd.DataFrame({
    "Id": test_ids,
    "Predicted_SalePrice": final_predictions
})

display(prediction_summary.head(10))

print("\nPrediction Summary:")
print(prediction_summary["Predicted_SalePrice"].describe())

,Id,Predicted_SalePrice
0,1461,125763.508
1,1462,162952.164
2,1463,183611.082
3,1464,186284.340
4,1465,196178.534
5,1466,182321.464
6,1467,176294.200
7,1468,175421.794
8,1469,187739.434
9,1470,130890.280



Prediction Summary:
count      1459.000000
mean     178702.980683
std       73893.709340
min       62550.410000
25%      129796.584000
50%      159144.950000
75%      208805.129000
max      579629.416000
Name: Predicted_SalePrice, dtype: float64


# To Get Info :
Mean predicted price



Minimum predicted price



Maximum predicted price



Standard deviation

In [11]:
# Final Model Save

joblib.dump(
    selected_model,
    "../models/final_house_price_model.pkl"
)

print("Final Model Saved Successfully!")

Final Model Saved Successfully!


In [12]:
# Make Kaggle Submission File

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": final_predictions
})

submission.head()

,Id,SalePrice
0,1461,125763.508
1,1462,162952.164
2,1463,183611.082
3,1464,186284.340
4,1465,196178.534


In [13]:
# To Make CSV file :
submission.to_csv(
    "../data/processed/submission.csv",
    index=False
)

print("Submission File Created Successfully!")

Submission File Created Successfully!


In [14]:
# Final Validation Of Submission File , And To Check 

submission_check = pd.read_csv(
    "../data/processed/submission.csv"
)

print("Submission Shape:", submission_check.shape)

print("\nColumn Names:")
print(submission_check.columns.tolist())

print("\nMissing Values:")
print(submission_check.isnull().sum())

print("\nDuplicate IDs:")
print(submission_check["Id"].duplicated().sum())

print("\nNegative Predictions:")
print((submission_check["SalePrice"] < 0).sum())

Submission Shape: (1459, 2)

Column Names:
['Id', 'SalePrice']

Missing Values:
Id           0
SalePrice    0
dtype: int64

Duplicate IDs:
0

Negative Predictions:
0
